# Managed Agent Memory Toolkit: Demo (REST API)

This demo drives the full memory lifecycle for a personal travel assistant
entirely through the **AMT REST service** over HTTP:

> ingest -> extract (facts + **episodes**) -> reconcile -> summarize -> retrieve

```
┌─────────────────────────────────────────────────────────────┐
│                        Client Layer                         │
│                Copilot  •  Claude  •  Cursor                │
│                  Direct REST / SDK Callers                  │
└─────────────────────────────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│               Inference Platform API Gateway                │
│             OAuth 2.1  •  Entra ID  •  API Keys             │
│               Quotas  •  Metering  •  Logging               │
└─────────────────────────────────────────────────────────────┘
                               │
                               ▼
┌────────────────────────────┐   ┌────────────────────────────┐
│          REST API          │   │        MCP Endpoint        │
│      (Standard HTTP)       │   │     (Streamable HTTP)      │
└────────────────────────────┘   └────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                     Core Memory Service                     │
│                        (Shared SDK)                         │
└─────────────────────────────────────────────────────────────┘
                               │
                               ▼
┌────────────────────────────┐   ┌────────────────────────────┐
│      Azure Cosmos DB       │ ↔ │     Durable Functions      │
└────────────────────────────┘   └────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                      Azure AI Foundry                       │
└─────────────────────────────────────────────────────────────┘
```

*Jordan* plans and takes a Portugal trip across **three sessions spread over
weeks**. Two things happen along the way:

1. **Facts evolve** - budget, diet, and seat preference change - so
   contradictions accumulate and get resolved by a **reconcile** pass.
2. **Experiences happen** - a booking flow and a mid-trip flight cancellation -
   which the backend captures as first-class **episodes**: bounded experiences
   with an event timeline, an outcome, and lessons.

### How this differs from the SDK notebook

- **Cadence lives on the server.** You send **turns**; the deployed service
  extracts facts/episodes/summaries in the background on its own cadence
  (`MEMORY_PROCESSOR_OWNER=durable`). We **poll** until results appear.
- **Episodes are boundary-based and per-thread.** An episode closes when a
  conversation segment ends - here, an **idle time-gap** of more than ~30 min
  between turns. We put all turns in **one thread** and stamp each turn's
  `created_at` so the three sessions sit days apart; the two *completed* sessions
  (the ones a later session closes) become episodes. Episodes are extracted only
  from **bounded experiences**, not from isolated preference statements.
- **Search returns the current view.** Superseded facts are excluded, so
  retrieval reflects the up-to-date Jordan.

### Endpoints used

| Step | Endpoint |
|---|---|
| Ingest a turn | `POST /users/{uid}/threads/{tid}/memory` |
| Search facts (+ episodes) | `POST /users/{uid}/search` |
| Search episodes | `POST /users/{uid}/search/episodes` |
| Reconcile | `POST /users/{uid}/reconcile` |
| Generate / get thread summary | `POST` / `GET /users/{uid}/threads/{tid}/summary` |
| Generate / get user profile | `POST` / `GET /users/{uid}/summary` |


## 1. Setup

Point `BASE_URL` at the running service. This demo uses only the Python standard library (no extra packages).

In [9]:
import json
import os
import time
import uuid
import urllib.request
import urllib.error

# AMT REST service base URL. Override with AMT_REST_BASE_URL. Behind the
# InferencePlatform gateway the path is prefixed with /inference/memory.
BASE_URL = os.environ.get(
    "AMT_REST_BASE_URL",
    "<Add the url here>",
).rstrip("/")


def call(method, path, body=None, timeout=90):
    """Minimal JSON HTTP helper. Returns (status_code, parsed_json)."""
    url = f"{BASE_URL}{path}"
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, method=method)
    req.add_header("Content-Type", "application/json")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            payload = resp.read().decode()
            return resp.status, (json.loads(payload) if payload else None)
    except urllib.error.HTTPError as exc:
        payload = exc.read().decode()
        try:
            return exc.code, json.loads(payload)
        except Exception:
            return exc.code, payload


status, health = call("GET", "/health")
print(f"Health: HTTP {status} -> {health}")

# One thread for the whole trip (episodes segment within a thread), and a fresh
# user id per run so reruns do not mix state.
USER_ID = f"jordan-{uuid.uuid4().hex[:6]}"
THREAD_ID = "portugal-trip"
print(f"Base URL: {BASE_URL}")
print(f"Demo user: {USER_ID}   thread: {THREAD_ID}")

Health: HTTP 200 -> {'status': 'ok', 'processor_owner': 'durable'}
Base URL: <Add the url here>
Demo user: jordan-253820   thread: portugal-trip


## Cadence - how the backend fires

Memory processing runs on the **deployed service** (the durable Function app),
not this client, so there are no knobs to set here. The cadence below governs
when each step fires as turns are ingested - shown for reference so you can
follow along.

In [10]:
# Deployed AMT cadence (server-side; shown here for reference only).
CADENCE = {
    "FACT_EXTRACTION_EVERY_N": 2,      # extract facts every 2 turns
    "EPISODE_EVAL_EVERY_N": 4,         # evaluate an episode boundary every 4 turns
    "EPISODE_IDLE_GAP_SECONDS": 1800,  # a gap > 30 min (via created_at) closes the open episode
    "EPISODE_MIN_TURNS": 2,            # minimum turns before a natural boundary can close
    "EPISODE_MAX_TURNS": 40,           # force a boundary after 40 turns in one segment
    "EPISODE_TOPIC_DRIFT": 0,          # topic-drift boundary disabled (idle-gap + max-size only)
    "THREAD_SUMMARY_EVERY_N": 10,      # thread (session) summary every 10 turns
    "USER_SUMMARY_EVERY_N": 20,        # cross-thread user profile every 20 turns
    "DEDUP_EVERY_N": 5,                # reconcile every 5 fact batches (= every 10 turns)
}
print("Deployed AMT cadence (server-side):\n")
for _name, _value in CADENCE.items():
    print(f"  {_name:26} = {_value}")

Deployed AMT cadence (server-side):

  FACT_EXTRACTION_EVERY_N    = 2
  EPISODE_EVAL_EVERY_N       = 4
  EPISODE_IDLE_GAP_SECONDS   = 1800
  EPISODE_MIN_TURNS          = 2
  EPISODE_MAX_TURNS          = 40
  EPISODE_TOPIC_DRIFT        = 0
  THREAD_SUMMARY_EVERY_N     = 10
  USER_SUMMARY_EVERY_N       = 20
  DEDUP_EVERY_N              = 5


## 2. Helpers

`write_session` posts a session's turns **with backdated `created_at`
timestamps** so the sessions sit days apart in one thread. `list_facts` and
`list_episodes` read state directly via the **list endpoints** (no search query
needed); `search_facts` is kept for the semantic Q&A later. `wait_for` polls the
asynchronous backend until a condition is met.

In [11]:
def post_turn(role, content, created_at):
    status, out = call("POST", f"/users/{USER_ID}/threads/{THREAD_ID}/memory",
                       {"role": role, "content": content, "created_at": created_at})
    return status, out


def write_session(label, day, transcript):
    """Post a session's turns on a given ISO date, one minute apart. The >30 min
    gap between sessions (different days) is what lets the backend close each
    completed session into an episode."""
    print(f"=== {label} ({day}): posting {len(transcript)} turns ===")
    for i, (role, content) in enumerate(transcript):
        created_at = f"{day}T09:{i:02d}:00Z"
        status, _ = post_turn(role, content, created_at)
        flag = "ok" if status == 201 else f"HTTP {status}"
        print(f"  [{role:>5}] {content[:76]}  ({flag})")


def search_facts(query, top_k=10):
    status, out = call("POST", f"/users/{USER_ID}/search",
                       {"query": query, "top_k": top_k, "include_episodes": False})
    return out.get("items", []) if (status == 200 and out) else []


def list_facts(thread_id=None):
    """List the user's facts directly - no search query needed.
    GET /users/{uid}/memories?memory_types=fact (optionally scoped to a thread)."""
    path = f"/users/{USER_ID}/memories?memory_types=fact"
    if thread_id:
        path += f"&thread_id={thread_id}"
    status, out = call("GET", path)
    return out.get("items", []) if (status == 200 and out) else []


def show_facts(title="Active facts"):
    items = list_facts()
    print(f"\n{title} ({len(items)}):")
    for f in items:
        tags = f.get("tags") or []
        source = "agent" if "sys:agent-fact" in tags else "user"
        conf = f.get("confidence")
        note = f"conf {conf}, {source}" if conf is not None else source
        print(f"  - {f.get('content', '')}   ({note})")
    # return items


def list_episodes(thread_id=None):
    """List the user's episodes, newest first - no search query needed.
    GET /users/{uid}/episodes (optionally scoped to a thread)."""
    path = f"/users/{USER_ID}/episodes"
    if thread_id:
        path += f"?thread_id={thread_id}"
    status, out = call("GET", path)
    return out.get("items", []) if (status == 200 and out) else []


def wait_for(fn, label, timeout=240, poll=12):
    """Poll fn() until it returns truthy, or timeout. Extraction is asynchronous
    (durable Functions), so this can take a minute or two."""
    start = time.time()
    while True:
        val = fn()
        elapsed = int(time.time() - start)
        print(f"  [{elapsed:>3}s] {label}: {val}")
        if val or elapsed >= timeout:
            return val
        time.sleep(poll)

## 3. Session 1 - Booking the trip (a completed experience)

Jordan books the trip. This session states preferences (**facts**: budget
$2000, window seat, vegetarian) **and** completes a booking flow - a bounded
experience with an outcome, which will become an **episode**.

In [12]:
SESSION_1 = [
    ("user", "Help me book my Portugal trip for September. Budget around $2000, I prefer window seats and I am vegetarian."),
    ("agent", "I searched flights and booked TAP flight TP204 to Lisbon on Sep 12 for $780, window seat, vegetarian meal. Confirmation TP-9XK2."),
    ("user", "Great, also book a hotel near Alfama."),
    ("agent", "Booked Memmo Alfama for 5 nights, refundable, $640 total. You are all set."),
]

write_session("Session 1 - booking", "2026-07-01", SESSION_1)
print("\nWaiting for facts to be extracted...")
wait_for(lambda: len(list_facts()) >= 3, "facts visible")
show_facts("Active facts after Session 1")

=== Session 1 - booking (2026-07-01): posting 4 turns ===
  [ user] Help me book my Portugal trip for September. Budget around $2000, I prefer w  (ok)
  [agent] I searched flights and booked TAP flight TP204 to Lisbon on Sep 12 for $780,  (ok)
  [ user] Great, also book a hotel near Alfama.  (ok)
  [agent] Booked Memmo Alfama for 5 nights, refundable, $640 total. You are all set.  (ok)

Waiting for facts to be extracted...
  [  0s] facts visible: False
  [ 12s] facts visible: True

Active facts after Session 1 (6):
  - For the user's Portugal trip in September, the user's budget is around $2000.   (conf 1, user)
  - The user prefers window seats.   (conf 1, user)
  - The user is vegetarian.   (conf 1, user)
  - The agent booked the user on TAP flight TP204 to Lisbon on Sep 12 for $780 with a window seat and a vegetarian meal; the confirmation code is TP-9XK2.   (conf 0.98, agent)
  - For the user's Portugal trip, the user needs a hotel near Alfama.   (conf 0.96, user)
  - The agent boo

### The API returns the full record

Each result is the **complete** memory document (only the raw embedding vector
and Cosmos system fields are stripped). Below is one fact's full JSON so you can
see everything available - provenance tags, confidence, salience, timestamps,
source ids, metadata. The rest of the demo **filters** this down to the fields
worth showing on screen.

In [13]:
import json as _json

_facts = list_facts()
if _facts:
    print("Full fact document returned by GET /users/{uid}/memories:\n")
    print(_json.dumps(_facts[0], indent=2))

Full fact document returned by GET /users/{uid}/memories:

{
  "id": "fact_a40f9bd6467ecd6ceff9ae12e053f8e6",
  "user_id": "jordan-253820",
  "thread_id": "portugal-trip",
  "role": "system",
  "type": "fact",
  "content": "For the user's Portugal trip in September, the user's budget is around $2000.",
  "metadata": {
    "category": "requirement",
    "temporal_context": "September",
    "source": "user"
  },
  "created_at": "2026-07-01T09:03:00+00:00",
  "updated_at": "2026-07-01T09:03:00+00:00",
  "tags": [
    "sys:auto-extracted",
    "sys:fact",
    "topic:budget",
    "topic:portugal",
    "topic:travel"
  ],
  "salience": 0.9,
  "confidence": 1,
  "content_hash": "7fb459d3257794f804e52ae29668db1e",
  "supersedes_ids": [],
  "source_memory_ids": [],
  "prompt_id": "extract_memories-v2.prompty",
  "prompt_version": "v4-additive",
  "source_fact_ids": [],
  "source_episodic_ids": [],
  "memory_type": "fact"
}


## 4. Session 2 - A mid-trip incident (another experience)

Weeks later, during the trip, a connecting flight is cancelled and the agent
rebooks Jordan - a second bounded experience (**episode**). Jordan also updates
two **facts**: budget -> $3500 and diet -> pescatarian. We wait until an updated
value is actually visible so the printout reflects the change, not a stale one.

In [14]:
SESSION_2 = [
    ("user", "I am at Lisbon airport and my connecting flight to Porto just got cancelled."),
    ("agent", "I rebooked you on the 6pm Alfa Pendular train to Porto and arranged a 40 euro meal voucher. You arrive by 9pm."),
    ("user", "Thanks. Update my total budget to $3500 and I have become pescatarian."),
    ("agent", "Noted budget $3500 and pescatarian going forward."),
]

write_session("Session 2 - trip incident", "2026-07-21", SESSION_2)
print("\nWaiting until an updated value (budget / diet) is visible...")
wait_for(lambda: any(("3500" in f.get("content", "") or "pescatarian" in f.get("content", "").lower())
                     for f in list_facts()), "updated value visible")
show_facts("Active facts after Session 2")

=== Session 2 - trip incident (2026-07-21): posting 4 turns ===
  [ user] I am at Lisbon airport and my connecting flight to Porto just got cancelled.  (ok)
  [agent] I rebooked you on the 6pm Alfa Pendular train to Porto and arranged a 40 eur  (ok)
  [ user] Thanks. Update my total budget to $3500 and I have become pescatarian.  (ok)
  [agent] Noted budget $3500 and pescatarian going forward.  (ok)

Waiting until an updated value (budget / diet) is visible...
  [  0s] updated value visible: False
  [ 12s] updated value visible: True

Active facts after Session 2 (11):
  - For the user's Portugal trip in September, the user's budget is around $2000.   (conf 1, user)
  - The user prefers window seats.   (conf 1, user)
  - The user is vegetarian.   (conf 1, user)
  - The agent booked the user on TAP flight TP204 to Lisbon on Sep 12 for $780 with a window seat and a vegetarian meal; the confirmation code is TP-9XK2.   (conf 0.98, agent)
  - For the user's Portugal trip, the user needs a h

## 5. Session 3 - Home, and a preference change

Back in Austin, Jordan switches to aisle seats after the long-haul and adds a
loyalty program. This is mostly preference updates (**facts**), and being the
latest session it stays "open" - so it is not itself episodized yet. Its later
timestamp is what closes **Session 2** into an episode.

In [15]:
SESSION_3 = [
    ("user", "Back home in Austin now after the trip."),
    ("agent", "Welcome back! How did everything go?"),
    ("user", "That 10-hour window seat was rough, switch me to aisle seats from now on, and save that I fly TAP as a frequent flyer."),
    ("agent", "Updated to aisle seats and added TAP frequent flyer to your profile."),
]

write_session("Session 3 - home", "2026-08-05", SESSION_3)
print("\nWaiting until the new fact (aisle / Austin / TAP) is visible...")
wait_for(lambda: any(("aisle" in f.get("content", "").lower() or "austin" in f.get("content", "").lower())
                     for f in list_facts()), "new fact visible")
show_facts("Active facts after Session 3")

=== Session 3 - home (2026-08-05): posting 4 turns ===
  [ user] Back home in Austin now after the trip.  (ok)
  [agent] Welcome back! How did everything go?  (ok)
  [ user] That 10-hour window seat was rough, switch me to aisle seats from now on, an  (ok)
  [agent] Updated to aisle seats and added TAP frequent flyer to your profile.  (ok)

Waiting until the new fact (aisle / Austin / TAP) is visible...
  [  0s] new fact visible: False
  [ 12s] new fact visible: True

Active facts after Session 3 (13):
  - For the user's Portugal trip in September, the user's budget is around $2000.   (conf 1, user)
  - The agent booked the user on TAP flight TP204 to Lisbon on Sep 12 for $780 with a window seat and a vegetarian meal; the confirmation code is TP-9XK2.   (conf 0.98, agent)
  - For the user's Portugal trip, the user needs a hotel near Alfama.   (conf 0.96, user)
  - The agent booked Memmo Alfama for 5 nights as a refundable hotel stay for $640 total.   (conf 0.97, agent)
  - On 2026-07-2

## 6. Episodes - the trip's bounded experiences

This is the episodic layer. The backend closed the two **completed** sessions
(booking, and the airport incident) into episodes: each has a title, an
event-time span, an ordered event timeline, an **outcome**, and lessons. Session
3 is still open, so it is not episodized. We read them with the **list endpoint**
`GET /users/{uid}/episodes` (newest first) - no search query needed.

*(Mechanic: the backend evaluates episode boundaries every few turns, and a
segment closes only once a later turn reveals the >30 min gap - so the demo uses
three equal-length sessions to give the incident episode a boundary to close
against.)*

In [16]:
episodes = list_episodes()

print(f"\n{len(episodes)} episode(s):\n")
for e in episodes:
    print(f"* {e.get('title', '(untitled)')}")
    print(f"    when      : {e.get('started_at')}  ->  {e.get('ended_at')}")
    if e.get("participants"):
        print(f"    people    : {', '.join(e['participants'])}")
    oc = e.get("outcome") or {}
    if oc:
        print(f"    outcome   : {oc.get('status')} - {oc.get('description', '')}")
    for ev in e.get("events", []):
        print(f"      {ev.get('sequence')}. {ev.get('description', '')}")
    if e.get("lessons"):
        for lesson in e["lessons"]:
            print(f"    lesson    : {lesson}")
    print(f"    salience  : {e.get('salience')}   confidence: {e.get('confidence')}")
    print()


2 episode(s):

* Rebooked Porto connection after Lisbon flight cancellation
    when      : 2026-07-21T09:00:00+00:00  ->  2026-07-21T09:01:00+00:00
    outcome   : successful - The cancelled flight connection was replaced with a confirmed 6pm train to Porto plus a 40 euro meal voucher, with arrival by 9pm.
      1. The user reported being at Lisbon airport and that the connecting flight to Porto had just been cancelled.
      2. The agent rebooked the user on the 6pm Alfa Pendular train to Porto, arranged a 40 euro meal voucher, and said the user would arrive by 9pm.
    salience  : 0.89   confidence: 0.98

* Booked Portugal flight and Lisbon hotel for September trip
    when      : 2026-07-01T09:00:00+00:00  ->  2026-07-01T09:03:00+00:00
    outcome   : successful - The Portugal trip booking was completed with a flight to Lisbon and a 5-night hotel reservation near Alfama within the stated budget context.
      1. The user requested help booking a Portugal trip for September with a 

## 7. Reconcile - resolve the contradictions

Jordan changed budget, diet, and seat over the trip, so both old and new values
were extracted. `POST /reconcile` resolves each conflicting pair: it keeps the
current fact and **supersedes** the loser (soft delete, preserved for audit).
After this, search reflects a fully reconciled picture ($3500, pescatarian,
aisle).

In [17]:
status, stats = call("POST", f"/users/{USER_ID}/reconcile", {})
print(f"reconcile: HTTP {status} -> {stats}")

show_facts("Active facts AFTER reconcile")

reconcile: HTTP 200 -> {'kept': 7, 'merged': 0, 'contradicted': 1}

Active facts AFTER reconcile (12):
  - The agent booked the user on TAP flight TP204 to Lisbon on Sep 12 for $780 with a window seat and a vegetarian meal; the confirmation code is TP-9XK2.   (conf 0.98, agent)
  - For the user's Portugal trip, the user needs a hotel near Alfama.   (conf 0.96, user)
  - The agent booked Memmo Alfama for 5 nights as a refundable hotel stay for $640 total.   (conf 0.97, agent)
  - On 2026-07-21, the user was at Lisbon Airport and the user's connecting flight to Porto was cancelled.   (conf 1, user)
  - The agent rebooked the user on the 6 PM Alfa Pendular train to Porto on 2026-07-21, with arrival by 9 PM.   (conf 0.98, agent)
  - The agent arranged a 40 euro meal voucher for the user on 2026-07-21.   (conf 0.98, agent)
  - The user's total budget is $3500.   (conf 0.96, user)
  - The user has become pescatarian.   (conf 0.96, user)
  - On 2026-08-05, the user was back home in Austin aft

## 8. Retrieval - "What do you know about me?"

Two complementary reads:

- **List** (`GET /memories`) returns state directly - what we used above for
  `show_facts`, no query needed.
- **Search** (`POST /search`) ranks by relevance for a question, and can blend
  facts + episodes in one call (`include_episodes=true`).

Because reconcile superseded the stale values, both reflect the current Jordan.

In [18]:
print("Targeted questions (facts):\n")
for q in ["seat preference", "dietary restriction", "travel budget", "home city and airline"]:
    hits = search_facts(q, top_k=2)
    print(f"Q: {q}")
    for f in hits:
        print(f"   -> {f.get('content', '')[:100]}")
    print()

print("Blended recall (facts + episodes) for 'the Porto trip disruption':\n")
status, out = call("POST", f"/users/{USER_ID}/search",
                   {"query": "the Porto trip disruption and how it was handled", "top_k": 6,
                    "include_episodes": True})
for it in (out or {}).get("items", []):
    kind = it.get("memory_type")
    text = it.get("title") if kind == "episodic" else it.get("content", "")
    print(f"  [{kind:>13}] {text}")

Targeted questions (facts):

Q: seat preference
   -> From 2026-08-05 onward, the user asked to be switched to aisle seats instead of window seats for fli
   -> The agent updated the user's profile to prefer aisle seats and added TAP frequent flyer information.

Q: dietary restriction
   -> The user has become pescatarian.
   -> From 2026-08-05 onward, the user asked to be switched to aisle seats instead of window seats for fli

Q: travel budget
   -> The user's total budget is $3500.
   -> The agent booked Memmo Alfama for 5 nights as a refundable hotel stay for $640 total.

Q: home city and airline
   -> The agent updated the user's profile to prefer aisle seats and added TAP frequent flyer information.
   -> The user flies TAP as a frequent flyer.

Blended recall (facts + episodes) for 'the Porto trip disruption':

  [         fact] On 2026-07-21, the user was at Lisbon Airport and the user's connecting flight to Porto was cancelled.
  [     episodic] Rebooked Porto connection after

## 9. Summaries - thread recap

The **thread summary** recaps the whole trip thread. It is generated
on cadence, and the REST API also lets you generate and read them on demand.

In [19]:
# Thread summary (generate + read back).
status, ts = call("POST", f"/users/{USER_ID}/threads/{THREAD_ID}/summary", {})
print(f"POST thread summary: HTTP {status}\n")
print("Trip thread summary:\n ", ts.get("content", "")[:400])

POST thread summary: HTTP 201

Trip thread summary:
  The conversation was about planning and managing the user's Portugal trip in September, including booking travel and lodging, handling a disruption during the trip, and updating the user's travel preferences afterward. The trip was booked with a TAP flight to Lisbon, a hotel near Alfama, and a replacement train to Porto after a cancelled connection; the user's budget and meal and seat preferences 


## 10. User Profile/Summary

The **user summary** is the cross-thread profile rolled up from Jordan's reconciled facts. It is generated
on cadence, and the REST API also lets you generate and read them on demand.

In [20]:
# User profile (generate + read back).
call("POST", f"/users/{USER_ID}/summary", {})
status, us = call("GET", f"/users/{USER_ID}/summary")
print(f"\nGET user summary: HTTP {status}\n")
print("Jordan's profile\n")
print(us.get("content", ""))
for section, value in (us.get("structured_summary") or {}).items():
    if not value:
        continue
    label = section.replace("_", " ").title()
    if isinstance(value, list):
        print(f"\n{label}:")
        for item in value:
            print(f"  - {item}")
    elif isinstance(value, dict):
        print(f"\n{label}:")
        for k, v in value.items():
            if v:
                print(f"  - {k}: {v}")
    else:
        print(f"\n{label}: {value}")


GET user summary: HTTP 200

Jordan's profile

As of 2026-08-05, the user was back home in Austin.; The user flies TAP as a frequent flyer.

Key Facts:
  - As of 2026-08-05, the user was back home in Austin.
  - The user flies TAP as a frequent flyer.

Personal Preferences:
  - As of 2026-08-05, the user prefers aisle seats for flights instead of window seats.
  - The user is pescatarian and should be given pescatarian-compatible meal options for travel and reservations.

Topics:
  - travel
  - portugal
  - flights
  - hotels
  - rail-travel
  - tap-air-portugal
  - seat-preferences
  - meal-preferences
  - austin


## 11. Recap

**What we showed, end to end, over REST:**

1. **Ingest**: posted 3 time-gapped sessions into one trip thread.
2. **Facts (async)**: the durable backend extracted facts; contradictions
   (budget / diet / seat) accumulated.
3. **List**: read facts and episodes directly with `GET /memories` and
   `GET /episodes` - no search query needed (`thread_id` optional on both).
4. **Episodes**: the two completed experiences - the booking and the flight
   cancellation - became first-class episodes with an event timeline, outcome,
   and lessons. The current (open) session was not episodized, by design.
5. **Reconcile**: `POST /reconcile` resolved the contradictions.
6. **Summarize**: generated the trip thread summary and Jordan's user profile.
7. **Search**: `POST /search` for relevance-ranked Q&A and blended
   facts+episodes recall.

**Why episodes needed experiences.** Episodes capture bounded *events* (a
booking, an incident), not isolated preferences - so the sessions were written
as things that happened, with outcomes. And because episodes are boundary-based
per thread, the turns share one thread with `created_at` timestamps days apart.
